In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Download directly to the Colab temporary disk
!wget -O /content/00-All.vcf.gz https://ftp.ncbi.nih.gov/snp/organisms/human_9606_b151_GRCh37p13/VCF/00-All.vcf.gz

--2026-06-26 18:22:09--  https://ftp.ncbi.nih.gov/snp/organisms/human_9606_b151_GRCh37p13/VCF/00-All.vcf.gz
Resolving ftp.ncbi.nih.gov (ftp.ncbi.nih.gov)... 130.14.250.31, 130.14.250.7, 2607:f220:41e:250::31, ...
Connecting to ftp.ncbi.nih.gov (ftp.ncbi.nih.gov)|130.14.250.31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15718889328 (15G) [application/x-gzip]
Saving to: ‘/content/00-All.vcf.gz’

/content/00-All.vcf 100%[===================>]  14.64G  30.0MB/s    in 5m 7s   

2026-06-26 18:27:17 (48.8 MB/s) - ‘/content/00-All.vcf.gz’ saved [15718889328/15718889328]



In [ ]:
import pandas as pd
import gzip
import os

# Updated paths based on your Colab environment and Google Drive layout
CLEANED_DIR = "/content/drive/MyDrive"
DBSNP_FILE  = "/content/00-All.vcf.gz"

def get_cad_rsids(cleaned_dir):
    """Extracts a set of all unique rsIDs from the CAD files to save memory."""
    print("Extracting target rsIDs from CAD files...")
    target_rsids = set()
    files_to_check = ["CAD_both_clean.tsv.gz", "CAD_female_clean.tsv.gz", "CAD_male_clean.tsv.gz"]

    for fname in files_to_check:
        fpath = os.path.join(cleaned_dir, fname)
        if os.path.exists(fpath):
            # Only load the rsid column to save RAM
            df = pd.read_csv(fpath, compression="gzip", usecols=["rsid"])
            target_rsids.update(df["rsid"].tolist())

    print(f"Total unique rsIDs to match: {len(target_rsids):,}")
    return target_rsids

def load_dbsnp_filtered(dbsnp_file, target_rsids):
    """Loads only VCF records that match our CAD rsIDs to prevent RAM crashes."""
    print("Scanning 15GB dbSNP file (this will take a few minutes)...")
    records = []

    with gzip.open(dbsnp_file, 'rt') as f:
        for line in f:
            if line.startswith("#"):
                continue

            # Split only the first 5 columns to be faster
            parts = line.split("\t", 5)
            if len(parts) < 5:
                continue

            rsid = parts[2]
            # Only keep the record if it's in our CAD files
            if rsid in target_rsids:
                ref = parts[3]
                alt = parts[4].split(",")[0]
                records.append({"rsid": rsid, "REF": ref, "ALT": alt})

    print(f"Found {len(records):,} matching records in dbSNP.")
    return pd.DataFrame(records)

def add_nea(cleaned_file, dbsnp_df):
    """Merges dbSNP data and applies NEA logic."""
    print(f"\nApplying NEA to {os.path.basename(cleaned_file)}...")
    df = pd.read_csv(cleaned_file, compression="gzip")

    # Merge on rsid
    merged = df.merge(
        dbsnp_df[["rsid", "REF", "ALT"]],
        on="rsid", how="left"
    )

    # Fill NEA where missing
    mask = merged["NEA"] == "."
    merged.loc[mask, "NEA"] = merged.loc[mask].apply(
        lambda r: r["ALT"] if r["reference_allele"] == r["REF"]
                  else r["REF"] if r["reference_allele"] == r["ALT"]
                  else ".", axis=1
    )

    # Clean up and save back to Drive
    merged = merged.drop(columns=["REF","ALT"])
    merged.to_csv(cleaned_file, index=False, compression="gzip")

    filled = (merged["NEA"] != ".").sum()
    still_missing = (merged["NEA"] == ".").sum()
    print(f"  NEA filled: {filled:,} | Still missing: {still_missing:,}")

# --- PIPELINE EXECUTION ---

# 1. Get the list of rsIDs we actually need
target_rsids = get_cad_rsids(CLEANED_DIR)

# 2. Extract only those rsIDs from the giant dbSNP file
dbsnp_filtered_df = load_dbsnp_filtered(DBSNP_FILE, target_rsids)

# 3. Apply the logic to each CAD file in your Drive
files_to_process = ["CAD_both_clean.tsv.gz", "CAD_female_clean.tsv.gz", "CAD_male_clean.tsv.gz"]
for fname in files_to_process:
    fpath = os.path.join(CLEANED_DIR, fname)
    if os.path.exists(fpath):
        add_nea(fpath, dbsnp_filtered_df)
    else:
        print(f"Warning: Could not find {fname} in {CLEANED_DIR}")

print("\nPhase 2 Complete! Your files in Google Drive have been overwritten with the updated NEA columns.")

Extracting target rsIDs from CAD files...
Total unique rsIDs to match: 7,422,061
Scanning 15GB dbSNP file (this will take a few minutes)...
Found 7,415,940 matching records in dbSNP.

Applying NEA to CAD_both_clean.tsv.gz...
  NEA filled: 6,599,136 | Still missing: 827,627

Applying NEA to CAD_female_clean.tsv.gz...
  NEA filled: 6,599,136 | Still missing: 827,627

Applying NEA to CAD_male_clean.tsv.gz...
  NEA filled: 6,599,136 | Still missing: 827,627

Phase 2 Complete! Your files in Google Drive have been overwritten with the updated NEA columns.



Processing: CAD_both_clean.tsv.gz
  SNPs with NEA = '.': 827,627 / 7,426,763
  Loading 826,737 rsIDs from dbSNP …


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/00-All.vcf.gz'